In [78]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [52]:
data = {'sepal length': [5.1, 4.9, 4.7, 4.6, 5],
        'sepal width': [3.5, 3, 3.2, 3.1, 3.6]}

In [53]:
df = pd.DataFrame(data)
df

,sepal length,sepal width
0,5.1,3.5
1,4.9,3.0
2,4.7,3.2
3,4.6,3.1
4,5.0,3.6


## 1. Standardise the data.
1.1 Work out the standard deviation

1.2 Work out the mean

1.3 Complete the z standardisation

In [54]:
x1 = df['sepal length']
x1m = x1.mean()

x1_centered = x1.map(lambda x: x - x1m)
x1_centered


0    0.24
1    0.04
2   -0.16
3   -0.26
4    0.14
Name: sepal length, dtype: float64

In [55]:
# Standard deviation of x1
x1std = x1.std()
x1std

np.float64(0.2073644135332772)

In [56]:
s1 = pd.Series(map(lambda x: (x - x1m)/x1std, x1), dtype='double[pyarrow]', name = 'Z sepal length')
s1

0    1.157383
1    0.192897
2   -0.771589
3   -1.253831
4     0.67514
Name: Z sepal length, dtype: double[pyarrow]

In [57]:
x2 = df['sepal width']
x2m = x2.mean()
x2std = x2.std()

s2 = pd.Series(x2.map(lambda x: (x - x2m)/ x2std))

frames = [s1, s2]
dfz = pd.concat(frames, axis=1)


In [58]:
dfz

,Z sepal length,sepal width
0,1.157383,0.849934
1,0.192897,-1.081734
2,-0.771589,-0.309067
3,-1.253831,-0.695401
4,0.67514,1.236268


## 2. Compute the covariance matrix.

In [59]:
# Since this is the covariance matrix of standardised data, this is technically a correlation matrix
dfz_cov = dfz.cov()
dfz_cov

,Z sepal length,sepal width
Z sepal length,1.000000,0.680019
sepal width,0.680019,1.000000


## 3. Diagonalise the covariance matrix.

In [60]:
eigenvalues, eigenvectors = np.linalg.eig(dfz_cov)
eigenvalues

array([1.68001929, 0.31998071])

## 4. Find the percentage contribution to the trace of each eigenvalue.

In [61]:
explained_variance = eigenvalues / eigenvalues.sum()
explained_variance

array([0.84000965, 0.15999035])

## 5. Verify that the eigenvectors are orthogonal.

In [62]:
eigenvectors

array([[ 0.70710678, -0.70710678],
       [ 0.70710678,  0.70710678]])

In [63]:
# The two different eigenvectors dot product should be 0
np.round(eigenvectors.T @ eigenvectors)

array([[ 1., -0.],
       [-0.,  1.]])

In [64]:
identity = np.eye(eigenvectors.shape[0])
is_orthonormal = np.allclose(eigenvectors.T @ eigenvectors, identity)
is_orthonormal

True

### 5.1 Manual orthonormality check

In [65]:
# The length of the eigenvector should be one
# Note: it should be the same eigenvector times by itself
v1 = eigenvectors[:,0]
(v1.T @ v1)**0.5

np.float64(1.0)

In [66]:
# The length of the eigenvector should be one
# Note: it should be the same eigenvector times by itself
v2 = eigenvectors[:,1]
(v2.T @ v2)**0.5

np.float64(1.0)

In [67]:
# The two different eigenvectors dot product should be 0
v1.T @ v2

np.float64(0.0)

## 6. Rewrite the standardised coordinates using the eigenvectors as a basis.

dfz is a matrix 5x2
eigenvectors is a matrix 2x2

so: 5x2 matrix @ 2x2 matrix

In [82]:
dfcoord = dfz @ eigenvectors
dfcoord = pd.DataFrame(dfcoord.values, columns=['PC1', 'PC2'])
dfcoord

,PC1,PC2
0,1.419387,-0.217399
1,-0.628503,-0.901301
2,-0.764139,0.327052
3,-1.378315,0.39487
4,1.351569,0.396777
